# Step 4 - Equation Parsing & Solving

Take the CNN predictions from step 3 and turn them into an actual equation we can solve.

Main challenges:
- `=` sign wasn't in HASYv2, so the CNN can't recognize it. We detect it by finding two `-` predictions that are vertically stacked.
- `x`, `X`, and `times` all look similar. Need context to tell if it's multiplication or a variable.
- Implicit multiplication - `2x` means `2*x`, need to insert the `*` automatically.

In [ ]:
import sys
sys.path.insert(0, '../src')

from solver import (
    detect_equals, resolve_ambiguity, 
    build_equation, solve_equation, solve_from_predictions
)

## Equals sign detection

The `=` sign gets segmented as two separate `-` bars by our contour-based segmentation. The CNN sees each bar individually and predicts `-` for both.

Fix: if two consecutive `-` predictions have similar x positions (vertically stacked), merge them into `=`.

In [ ]:
# simulate what happens with '2x + 3 = 7'
# the '=' would be two '-' bars at roughly the same x position

fake_predictions = [
    ('2', 0.99), ('x', 0.85), ('+', 0.97), ('3', 0.98),
    ('-', 0.90), ('-', 0.88),  # these two are the '=' sign
    ('7', 0.99)
]

# bounding boxes (x, y, w, h) - the two '-' bars are at similar x but different y
fake_boxes = [
    (10, 30, 25, 35),   # 2
    (45, 30, 25, 35),   # x
    (85, 25, 20, 30),   # +
    (120, 30, 25, 35),  # 3
    (160, 35, 30, 8),   # top bar of =
    (162, 50, 28, 8),   # bottom bar of =
    (210, 30, 25, 35),  # 7
]

preds, boxes = detect_equals(fake_predictions, fake_boxes)
print('before:', [p[0] for p in fake_predictions])
print('after: ', [p[0] for p in preds])
print(f'\nmerged two \'-\' into \'=\' - went from {len(fake_predictions)} to {len(preds)} symbols')

## x/X/times ambiguity

The CNN confuses lowercase `x`, uppercase `X`, and the multiplication sign `times` (they all look like an X shape). From the confusion matrix in step 3, lowercase `x` had 0% F1 score.

Strategy:
- `times` prediction -> always treat as `*`
- `X` between two operands -> treat as `*` (e.g. `3 X 4` -> `3 * 4`)
- Everything else -> treat as variable `x`
- `div` -> `/`

In [ ]:
# test ambiguity resolution
test_cases = [
    # 2x + 3 = 7 (x is a variable here)
    [('2', .99), ('x', .7), ('+', .98), ('3', .99), ('=', .95), ('7', .99)],
    # 3 * 4 + 2 (times is multiplication)
    [('3', .99), ('times', .8), ('4', .99), ('+', .98), ('2', .99)],
    # 5 X 2 (X between digits = multiply)
    [('5', .99), ('X', .6), ('2', .99)],
    # x + 1 = 5 (x at start = variable)
    [('x', .7), ('+', .99), ('1', .99), ('=', .95), ('5', .99)],
    # 8 / 2 + 1
    [('8', .99), ('div', .9), ('2', .99), ('+', .98), ('1', .99)],
]

for preds in test_cases:
    resolved = resolve_ambiguity(preds)
    before = ''.join(p[0] for p in preds)
    after = ''.join(p[0] for p in resolved)
    print(f'{before:20s} -> {after}')

## Building equation string

After resolving ambiguities, build the equation string. The main thing here is inserting implicit multiplication - when a digit is right next to a variable (`2x` -> `2*x`).

In [ ]:
# test equation building
test_preds = [
    [('2', .99), ('x', .85), ('+', .97), ('3', .98), ('=', .95), ('7', .99)],
    [('3', .99), ('*', .80), ('4', .99), ('+', .98), ('2', .99)],
    [('x', .80), ('+', .99), ('1', .99), ('=', .95), ('5', .99)],
    [('2', .99), ('x', .85), ('-', .97), ('5', .99)],
]

for preds in test_preds:
    eq = build_equation(preds)
    print(f'symbols: {[p[0] for p in preds]:30s} -> equation: {eq}')

## Solving with SymPy

Now the actual solving. SymPy handles:
- Pure arithmetic: `3+4` -> `7`
- Linear equations: `2*x+3=7` -> `x=2`
- Verification: `3+4=7` -> `True`

In [ ]:
# test solving different types of equations
test_equations = [
    '2*x+3=7',       # linear equation
    '3+4',           # arithmetic
    '3+4=7',         # verification (true)
    '3+4=8',         # verification (false)
    '8/2+1',         # arithmetic with division
    'x+1=5',         # simple linear
    '3*x-9=0',       # linear
    '5*x+2=3*x+8',  # variable on both sides
]

for eq in test_equations:
    result = solve_equation(eq)
    if result['type'] == 'arithmetic':
        print(f'{eq:20s} = {result["result"]}')
    elif result['type'] == 'equation':
        print(f'{eq:20s} -> {result["variable"]} = {result["solutions"]}')
    elif result['type'] == 'verification':
        print(f'{eq:20s} -> {"TRUE" if result["result"] else "FALSE"}')
    else:
        print(f'{eq:20s} -> {result}')

## Full pipeline test

Test the complete flow: fake CNN output -> detect equals -> resolve ambiguity -> build equation -> solve.

In [ ]:
# simulate full pipeline for '2x + 3 = 7'
# this is what the CNN + segmentation would give us
raw_preds = [
    ('2', 0.99), ('X', 0.65),  # CNN confused x with X
    ('+', 0.97), ('3', 0.98),
    ('-', 0.90), ('-', 0.88),  # = sign as two bars
    ('7', 0.99)
]
raw_boxes = [
    (10, 30, 25, 35),
    (45, 30, 25, 35),
    (85, 25, 20, 30),
    (120, 30, 25, 35),
    (160, 35, 30, 8),
    (162, 50, 28, 8),
    (210, 30, 25, 35),
]

result = solve_from_predictions(raw_preds, raw_boxes)

print(f'raw CNN output:  {[p[0] for p in raw_preds]}')
print(f'after parsing:   {result["symbols"]}')
print(f'equation string: {result["expression"]}')
print()
if result['type'] == 'equation':
    print(f'solution: {result["variable"]} = {result["solutions"]}')
else:
    print(f'result: {result}')

In [ ]:
# test a few more cases
test_scenarios = [
    {
        'name': '5 + 3 (pure arithmetic, no equals)',
        'preds': [('5', .99), ('+', .98), ('3', .99)],
        'boxes': [(10,30,25,35), (50,25,20,30), (85,30,25,35)],
    },
    {
        'name': '3x - 9 = 0',
        'preds': [
            ('3', .99), ('x', .70), ('-', .97), ('9', .98),
            ('-', .85), ('-', .82),  # = sign
            ('0', .99)
        ],
        'boxes': [
            (10,30,25,35), (45,30,25,35), (80,25,20,30), (110,30,25,35),
            (150,35,30,8), (152,50,28,8),
            (200,30,25,35)
        ],
    },
    {
        'name': '8 / 2 + 1',
        'preds': [('8', .99), ('div', .90), ('2', .99), ('+', .98), ('1', .99)],
        'boxes': [(10,30,25,35), (50,25,20,30), (85,30,25,35), (120,25,20,30), (155,30,25,35)],
    },
]

for scenario in test_scenarios:
    result = solve_from_predictions(scenario['preds'], scenario['boxes'])
    print(f"--- {scenario['name']} ---")
    print(f'symbols: {result["symbols"]}')
    print(f'equation: {result["expression"]}')
    if result['type'] == 'arithmetic':
        print(f'result: {result["result"]}')
    elif result['type'] == 'equation':
        print(f'solution: {result["variable"]} = {result["solutions"]}')
    print()

## Notes

What works:
- Equals detection from two stacked minus bars
- x/X/times disambiguation using context (between operands = multiply, otherwise = variable)
- Implicit multiplication insertion
- Solving linear equations, pure arithmetic, and equation verification

Limitations:
- Only handles single-variable linear equations for now
- The `=` detection depends on the segmentation keeping the two bars as separate contours in the right order
- If the CNN gets a symbol totally wrong, the parser can't recover from that
- Parentheses `()` weren't in HASYv2 either, so no bracket support yet